# 07. 음성 대화: 의도, 슬롯, 상태, 안전 확인

음성 대화 모델은 ASR 결과를 그대로 실행하지 않습니다.
의도 분류, 슬롯 추출, 상태 관리, 위험 조건 확인이 필요합니다.


In [ ]:
from pathlib import Path
import json
import math
import random
import statistics
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception as exc:
    plt = None
    print("matplotlib을 불러오지 못했습니다:", exc)

ROOT = Path.cwd()
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

random.seed(42)
np.random.seed(42)

def save_json(name, obj):
    path = ARTIFACTS / name
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def display_df(df, n=20):
    return df.head(n)


In [ ]:
commands = pd.DataFrame([
    ("로봇 팔을 왼쪽 선반으로 이동해", "move", "left shelf"),
    ("문서 번호 A17을 읽어줘", "read_doc", "A17"),
    ("알람이 들리면 멈춰", "safety_rule", "alarm"),
    ("請求書の合計金額を読んで", "read_doc", "total"),
    ("stop if you hear an alarm", "safety_rule", "alarm"),
    ("move to the charging station", "move", "charging station"),
], columns=["utterance", "intent", "slot"])

def classify_intent(text):
    lower = text.lower()
    if any(k in lower for k in ["move", "이동", "station", "선반"]):
        return "move"
    if any(k in lower for k in ["문서", "읽", "read", "請求書", "金額"]):
        return "read_doc"
    if any(k in lower for k in ["alarm", "알람", "멈", "stop"]):
        return "safety_rule"
    return "unknown"

def extract_slot(text):
    import re
    m = re.search(r"[A-Z]\d+", text.upper())
    if m:
        return m.group(0)
    for key in ["alarm", "알람", "left shelf", "charging station", "total"]:
        if key in text.lower():
            return key
    if "金額" in text:
        return "total"
    return ""

commands["pred_intent"] = commands["utterance"].map(classify_intent)
commands["pred_slot"] = commands["utterance"].map(extract_slot)
commands["intent_ok"] = commands.intent == commands.pred_intent
commands


In [ ]:
class DialogueState:
    def __init__(self):
        self.safety_stop = False
        self.pending_action = None

    def handle(self, utterance, sound_event=None):
        intent = classify_intent(utterance)
        slot = extract_slot(utterance)
        if sound_event == "alarm":
            self.safety_stop = True
            self.pending_action = None
            return {"action": "stop", "reason": "alarm_detected", "intent": intent, "slot": slot}
        if self.safety_stop and intent == "move":
            self.pending_action = (intent, slot)
            return {"action": "ask_confirmation", "reason": "move_after_alarm", "intent": intent, "slot": slot}
        if intent == "move":
            return {"action": "execute_move", "target": slot}
        if intent == "read_doc":
            return {"action": "run_ocr_docqa", "target": slot}
        if intent == "safety_rule":
            return {"action": "update_policy", "trigger": slot}
        return {"action": "fallback"}

state = DialogueState()
scenario = [
    ("문서 번호 A17을 읽어줘", None),
    ("왼쪽 선반으로 이동해", "alarm"),
    ("계속 이동해", None),
]
pd.DataFrame([state.handle(u, s) for u, s in scenario])


## 논문 연결

대화 상태는 최종 파이프라인의 사용자 가치 부분입니다.
논문 실험에서는 `알람 감지 후 잘못된 실행을 얼마나 줄이는가`를 safety metric으로 둘 수 있습니다.
